## Modelo preditivo LSTM - Tech Challenge Fase 04

### Instalação e importação de bibliotecas

In [1]:
%pip install -q yfinance tensorflow

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error


from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import plotly.graph_objects as go

import joblib

### Carregamento dos dados

In [3]:
acao = 'WEGE3.SA'
data_inicio = '2018-01-01'
data_fim = '2025-12-31'

dados = yf.download(acao, start = data_inicio, end = data_fim)

[*********************100%***********************]  1 of 1 completed


In [4]:
dados = dados.stack(level='Ticker').reset_index()
dados

Price,Date,Ticker,Close,High,Low,Open,Volume
0,2018-01-02,WEGE3.SA,8.220531,8.269815,8.000397,8.000397,4812860
1,2018-01-03,WEGE3.SA,8.095682,8.197535,8.079254,8.181108,4652960
2,2018-01-04,WEGE3.SA,8.016828,8.187678,8.016828,8.141680,3317600
3,2018-01-05,WEGE3.SA,8.049681,8.098965,7.980684,8.089108,2552680
4,2018-01-08,WEGE3.SA,8.115392,8.164675,7.905114,8.052965,3346200
...,...,...,...,...,...,...,...
1983,2025-12-22,WEGE3.SA,47.660000,48.849998,47.180000,48.700001,6391700
1984,2025-12-23,WEGE3.SA,48.279999,48.279999,47.720001,47.810001,4886100
1985,2025-12-26,WEGE3.SA,48.750000,48.750000,47.980000,48.020000,1829300
1986,2025-12-29,WEGE3.SA,48.709999,49.060001,48.270000,48.990002,3845900


In [5]:
dados.columns.name = None
dados['Date'] = pd.to_datetime(dados['Date'], format='%Y/%m/%d')

In [6]:
dados

,Date,Ticker,Close,High,Low,Open,Volume
0,2018-01-02,WEGE3.SA,8.220531,8.269815,8.000397,8.000397,4812860
1,2018-01-03,WEGE3.SA,8.095682,8.197535,8.079254,8.181108,4652960
2,2018-01-04,WEGE3.SA,8.016828,8.187678,8.016828,8.141680,3317600
3,2018-01-05,WEGE3.SA,8.049681,8.098965,7.980684,8.089108,2552680
4,2018-01-08,WEGE3.SA,8.115392,8.164675,7.905114,8.052965,3346200
...,...,...,...,...,...,...,...
1983,2025-12-22,WEGE3.SA,47.660000,48.849998,47.180000,48.700001,6391700
1984,2025-12-23,WEGE3.SA,48.279999,48.279999,47.720001,47.810001,4886100
1985,2025-12-26,WEGE3.SA,48.750000,48.750000,47.980000,48.020000,1829300
1986,2025-12-29,WEGE3.SA,48.709999,49.060001,48.270000,48.990002,3845900


In [7]:
dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 1988 entries, 0 to 1987
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype        
---  ------  --------------  -----        
 0   Date    1988 non-null   datetime64[s]
 1   Ticker  1988 non-null   str          
 2   Close   1988 non-null   float64      
 3   High    1988 non-null   float64      
 4   Low     1988 non-null   float64      
 5   Open    1988 non-null   float64      
 6   Volume  1988 non-null   int64        
dtypes: datetime64[s](1), float64(4), int64(1), str(1)
memory usage: 108.8 KB


### Preparação dos dados

In [8]:
features = ['Open', 'High', 'Low', 'Close', 'Volume']
selecao_dados = dados[features].values

scaler = MinMaxScaler(feature_range=(0, 1))
data_padronizados = scaler.fit_transform(selecao_dados)

In [9]:
def criar_periodos_dados(dados, janela_tempo=60):
    X, y = [], []
    indice_fechamento = features.index('Close')

    for periodo in range(janela_tempo, len(dados)):
        X.append(dados[periodo-janela_tempo:periodo])
        y.append(dados[periodo, indice_fechamento])

    return np.array(X), np.array(y)

JANELA_TEMPO = 60
X, y = criar_periodos_dados(data_padronizados, JANELA_TEMPO)

In [10]:
qtd_dados_treino = int(0.8 * len(X))

X_treino, X_validacao = X[:qtd_dados_treino], X[qtd_dados_treino:]
y_treino, y_validacao = y[:qtd_dados_treino], y[qtd_dados_treino:]

In [11]:
X_treino.shape

(1542, 60, 5)

### Treinamento do modelo LSTM

In [12]:
modelo = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_treino.shape[1], X_treino.shape[2])),
    Dropout(0.2),

    LSTM(64, return_sequences=False),
    Dropout(0.2),

    Dense(1)
])

modelo.compile(
    optimizer='adam',
    loss='mse'
)

modelo.summary()

c:\Users\igor_\Documents\GitHub\lstm-stock-prediction-api\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 64)         │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 51,009 (199.25 KB)

 Trainable params: 51,009 (199.25 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

historico = modelo.fit(
    X_treino, y_treino,
    validation_data=(X_validacao, y_validacao),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
49/49 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - loss: 0.0129 - val_loss: 0.0073
Epoch 2/100
49/49 ━━━━━━━━━━━━━━━━━━━━ 3s 62ms/step - loss: 0.0024 - val_loss: 0.0024
Epoch 3/100
49/49 ━━━━━━━━━━━━━━━━━━━━ 3s 54ms/step - loss: 0.0021 - val_loss: 0.0057
Epoch 4/100
49/49 ━━━━━━━━━━━━━━━━━━━━ 3s 63ms/step - loss: 0.0019 - val_loss: 0.0017
Epoch 5/100
49/49 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - loss: 0.0019 - val_loss: 0.0023
Epoch 6/100
49/49 ━━━━━━━━━━━━━━━━━━━━ 3s 57ms/step - loss: 0.0017 - val_loss: 0.0028
Epoch 7/100
49/49 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - loss: 0.0017 - val_loss: 0.0019
Epoch 8/100
49/49 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - loss: 0.0015 - val_loss: 0.0039
Epoch 9/100
49/49 ━━━━━━━━━━━━━━━━━━━━ 3s 53ms/step - loss: 0.0016 - val_loss: 0.0035
Epoch 10/100
49/49 ━━━━━━━━━━━━━━━━━━━━ 3s 53ms/step - loss: 0.0016 - val_loss: 0.0027
Epoch 11/100
49/49 ━━━━━━━━━━━━━━━━━━━━ 3s 53ms/step - loss: 0.0016 - val_loss: 0.0035
Epoch 12/100
49/49 ━━━━━━━━━━━━━━━━━━━━ 3s 55ms/step

### Avaliação do modelo

In [14]:
y_pred = modelo.predict(X_validacao)

13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step


In [15]:
def inverter_dados_padronizados(fechamento_padronizado):
    fechamento = np.zeros((len(fechamento_padronizado), len(features)))
    fechamento[:, features.index('Close')] = fechamento_padronizado.flatten()
    return scaler.inverse_transform(fechamento)[:, features.index('Close')]

In [16]:
y_val_inv = inverter_dados_padronizados(y_validacao)
y_pred_inv = inverter_dados_padronizados(y_pred)

In [17]:
mae = mean_absolute_error(y_val_inv, y_pred_inv)
rmse = np.sqrt(mean_squared_error(y_val_inv, y_pred_inv))
mape = np.mean(np.abs((y_val_inv - y_pred_inv) / y_val_inv)) * 100

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape:.2f}%")

MAE:  1.2248
RMSE: 1.5795
MAPE: 2.78%


### Visualização dos dados

In [18]:
dates = dados['Date'].iloc[JANELA_TEMPO + qtd_dados_treino:]

df_plot = pd.DataFrame({
    'Date': dates.values,
    'Real': y_val_inv,
    'Previsto': y_pred_inv
})

In [19]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_plot['Date'],
    y=df_plot['Real'],
    mode='lines',
    name='Preço Real',
    line=dict(width=2)
))

fig.add_trace(go.Scatter(
    x=df_plot['Date'],
    y=df_plot['Previsto'],
    mode='lines',
    name='Preço Previsto',
    line=dict(width=2, dash='dash')
))

fig.update_layout(
    title='Preço Real vs Preço Previsto (LSTM)',
    xaxis_title='Data',
    yaxis_title='Preço de Fechamento',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

In [20]:
y_train_pred = modelo.predict(X_treino)

y_train_inv = inverter_dados_padronizados(y_treino)
y_train_pred_inv = inverter_dados_padronizados(y_train_pred)

dates_train = dados['Date'].iloc[JANELA_TEMPO:JANELA_TEMPO + qtd_dados_treino]


49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step


In [21]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=dates_train,
    y=y_train_inv,
    mode='lines',
    name='Treino - Real'
))

fig.add_trace(go.Scatter(
    x=dates_train,
    y=y_train_pred_inv,
    mode='lines',
    name='Treino - Previsto',
    line=dict(dash='dot')
))

fig.add_trace(go.Scatter(
    x=df_plot['Date'],
    y=df_plot['Real'],
    mode='lines',
    name='Validação - Real'
))

fig.add_trace(go.Scatter(
    x=df_plot['Date'],
    y=df_plot['Previsto'],
    mode='lines',
    name='Validação - Previsto',
    line=dict(dash='dash')
))

fig.update_layout(
    title='Treino e Validação — Real vs Previsto (LSTM)',
    xaxis_title='Data',
    yaxis_title='Preço de Fechamento',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()


In [22]:
df_plot['Erro'] = df_plot['Real'] - df_plot['Previsto']

In [23]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_plot['Date'],
    y=df_plot['Erro'],
    mode='lines',
    name='Erro'
))

fig.update_layout(
    title='Erro da Previsão ao Longo do Tempo',
    xaxis_title='Data',
    yaxis_title='Erro (Real - Previsto)',
    template='plotly_white'
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="red",
    line_width=4,
    annotation_text="Linha de erro nulo",
    annotation_position="top left"
)

fig.show()


### Salvando o modelo

In [ ]:
modelo.save('../models/modelo_lstm_wege3.keras')

In [ ]:
joblib.dump(scaler, '../models/scaler_lstm.pkl')

['scaler_lstm.pkl']

In [24]:
import joblib

# 1. Salvar o modelo (formato recomendado do Keras)
modelo.save('../models/modelo_lstm.keras')
print("Modelo salvo como 'modelo_lstm.keras'")

# 2. Salvar o scaler (essencial para normalizar os dados novos)
joblib.dump(scaler, '../models/scaler.joblib')
print("Scaler salvo como 'scaler.joblib'")

Modelo salvo como 'modelo_lstm.keras'
Scaler salvo como 'scaler.joblib'
